# Notebook 7 — Build Your *Own* Copilot  🛠️
### Build an AI Business Copilot 🏔️ · Optional capstone

This notebook is a **template**. Everything you learned across Notebooks 1–6 —
prompting, retrieval (RAG), and tool use — is assembled here into **one self-contained
copilot** that you can point at **any business you like**. No retail or Trailhead
assumptions: bring your own documents and your own data.

**How to use it**
1. Fill in the ✏️ cells (your company, your files).
2. Drop your documents and data into two folders.
3. Run top to bottom and start asking questions.

> 🐍 **No deep coding needed.** Only the cells marked **✏️ PLUG IN** are meant to be
> edited. You can generate your documents, data, and even edits to these cells with an AI
> assistant (Claude, ChatGPT) — just keep the file **formats** the same.

## 1. Setup

Install the libraries (works in Google Colab and Databricks) and enter your Claude API key.

In [ ]:
%pip install -q anthropic sentence-transformers faiss-cpu pypdf pandas numpy

Enter your API key. It's typed into a hidden box and kept only for this session — it is **never saved in the notebook**.

In [ ]:
import os, getpass

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Paste your Anthropic API key: ")

print("API key is set. ✅")

## 2. Describe your business  ✏️ PLUG IN

This is the heart of the template — tell the copilot **who it works for**. Change these to
anything: a coffee roaster, a clinic, a software company, a library. The rest of the
notebook adapts automatically.

In [ ]:
# ✏️ Your company — change these two lines to your own (real or fictional) business.
COMPANY_NAME = "Example Co"
COMPANY_DESCRIPTION = (
    "Example Co is a small online shop. (Replace this with one or two sentences "
    "describing what your business does and who it serves.)"
)

# --- Settings you can leave as-is ---
MODEL      = "claude-haiku-4-5"   # cheapest & fastest; try "claude-sonnet-5" for harder questions
MAX_TOKENS = 600                  # cap on reply length (smaller = cheaper/faster)
TOP_K      = 4                    # how many document chunks to pull in per question

DOCS_DIR = "my_docs"              # folder for your .md / .txt documents (policies, FAQs, guides)
DATA_DIR = "my_data"              # folder for your .csv data tables

print(f"Configured copilot for: {COMPANY_NAME}")

## 3. Add your files

Your copilot draws on two kinds of knowledge:

| Folder | Put here | Used for | Built in Notebook |
| --- | --- | --- | --- |
| `my_docs/` | **documents** as `.md` or `.txt` (policies, FAQs, how-tos) | policy / "how does X work" questions (**RAG**) | 2–3 |
| `my_data/` | **data tables** as `.csv` (one file per table) | questions about specific records (**tool use**) | 4 |

Run the next cell to create the two folders, then add your files:

- **In Google Colab:** open the 📁 **Files** panel on the left and drag your files into
  `my_docs` / `my_data` — or **uncomment** the upload lines in the cell below.
- **Locally / Databricks:** copy your files into those folders.

> 💡 The cell drops in a tiny **example** doc and table so the notebook runs out of the box.
> Delete those once you've added your own files.

In [ ]:
from pathlib import Path

Path(DOCS_DIR).mkdir(exist_ok=True)
Path(DATA_DIR).mkdir(exist_ok=True)

# Seed a tiny example so a first run works end-to-end. Delete these once you add your own.
if not any(Path(DOCS_DIR).iterdir()):
    Path(DOCS_DIR, "example_faq.md").write_text(
        "# Example FAQ (delete me)\n\n"
        "## Returns\n"
        "Customers may return unused items within 30 days of delivery for a full refund. "
        "Final-sale items cannot be returned.\n\n"
        "## Hours\n"
        "Support is available Monday to Friday, 9am–5pm.\n",
        encoding="utf-8",
    )
if not any(Path(DATA_DIR).iterdir()):
    Path(DATA_DIR, "example_items.csv").write_text(
        "item_id,name,category,price,in_stock\n"
        "A-100,Sample Item,Demo,19.99,12\n"
        "A-101,Another Item,Demo,49.00,0\n",
        encoding="utf-8",
    )

print("📄 Documents in", DOCS_DIR, "->", [p.name for p in Path(DOCS_DIR).iterdir()])
print("📊 Data files in", DATA_DIR, "->", [p.name for p in Path(DATA_DIR).iterdir()])

# --- Optional (Google Colab): upload files from your computer ---
# from google.colab import files
# for name, data in files.upload().items():
#     folder = DATA_DIR if name.lower().endswith(".csv") else DOCS_DIR
#     Path(folder, name).write_bytes(data)

### Optional — your documents are PDFs?

The document pipeline reads **text** (`.md` / `.txt`), not PDFs. If your files are PDFs,
put them in `my_docs/` and run this cell to convert each one to a `.md` text file (using
`pypdf`, exactly as shown in Notebook 2). Skip it if your docs are already text.

In [ ]:
from pypdf import PdfReader
from pathlib import Path

pdfs = list(Path(DOCS_DIR).glob("*.pdf"))
for pdf in pdfs:
    reader = PdfReader(str(pdf))
    text = "\n".join((page.extract_text() or "") for page in reader.pages)
    out = pdf.with_suffix(".md")
    out.write_text(text, encoding="utf-8")
    print(f"Converted {pdf.name} -> {out.name}  ({len(text)} characters)")

if not pdfs:
    print("No PDFs found in", DOCS_DIR, "- nothing to convert (your docs are already text).")

## 4. Load & preview your data

This loads every `.csv` in `my_data/` into a table. **Look at the preview** — this is your
chance to catch problems before building the copilot.

> ✅ **Quick consistency check.** If your tables reference each other (e.g. an `orders`
> table pointing at a `customers` table), make sure the linking IDs actually match across
> files. AI-generated data often *looks* fine but references records that don't exist —
> which makes the copilot answer wrong. Eyeball a couple of rows now.

In [ ]:
import pandas as pd
from pathlib import Path

TABLES = {p.stem: pd.read_csv(p) for p in sorted(Path(DATA_DIR).glob("*.csv"))}

if not TABLES:
    print("No .csv files found in", DATA_DIR, "- the copilot will answer from documents only.")
for name, df in TABLES.items():
    print(f"\n=== {name}   ({len(df)} rows) ===")
    print("columns:", list(df.columns))
    print(df.head(3).to_string(index=False))

## 5. Build the knowledge base  *(no edits needed)*

This turns your documents into a searchable index: split each document into small
**chunks**, convert each chunk into an **embedding** (a numeric fingerprint of its
meaning), and store them in a **FAISS** index for fast similarity search. This is
Notebooks 2–3, condensed.

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss, numpy as np
from pathlib import Path

def chunk(text, source, max_chars=700):
    """Split a document into roughly paragraph-sized chunks."""
    chunks, current = [], ""
    for para in text.split("\n\n"):
        para = para.strip()
        if not para:
            continue
        if current and len(current) + len(para) > max_chars:
            chunks.append({"source": source, "text": current.strip()})
            current = ""
        current += para + "\n\n"
    if current.strip():
        chunks.append({"source": source, "text": current.strip()})
    return chunks

paths = sorted(Path(DOCS_DIR).glob("*.md")) + sorted(Path(DOCS_DIR).glob("*.txt"))
CHUNKS = [c for p in paths for c in chunk(p.read_text(encoding="utf-8"), p.name)]

embedder = index = None
if CHUNKS:
    embedder = SentenceTransformer("all-MiniLM-L6-v2")
    vectors = embedder.encode([c["text"] for c in CHUNKS], normalize_embeddings=True)
    index = faiss.IndexFlatIP(vectors.shape[1])
    index.add(np.array(vectors, dtype="float32"))
    print(f"Indexed {len(CHUNKS)} chunks from {len(paths)} document(s). ✅")
else:
    print("No .md/.txt documents found in", DOCS_DIR, "- the copilot will use data only.")

## 6. Give your copilot access to your data  *(no edits needed)*

We give Claude **one general-purpose tool**, `lookup_rows`, that can search any of your
tables by matching a column to a value. Because it's generic, it works for *any* business —
Claude reads your table names and columns (below) and decides what to look up. This is the
tool-use idea from Notebook 4, made domain-agnostic.

In [ ]:
import json

def lookup_rows(table_name, column, value, max_rows=10):
    """Return rows from a table where `column` matches `value`
    (exact match first, then a case-insensitive partial match)."""
    if table_name not in TABLES:
        return {"error": f"No table '{table_name}'. Available: {list(TABLES)}"}
    df = TABLES[table_name]
    if column not in df.columns:
        return {"error": f"Table '{table_name}' has no column '{column}'. Columns: {list(df.columns)}"}

    col = df[column].astype(str).str.strip()
    target = str(value).strip()
    matches = df[col.str.lower() == target.lower()]
    if matches.empty:
        matches = df[col.str.contains(target, case=False, na=False, regex=False)]

    rows = []
    for record in matches.head(max_rows).to_dict("records"):
        rows.append({k: (None if pd.isna(v) else v) for k, v in record.items()})
    return {"count": len(rows), "rows": rows}

TOOL_FUNCTIONS = {"lookup_rows": lookup_rows}

if TABLES:
    schema = "\n".join(f"  - {n} ({len(df)} rows): {', '.join(df.columns)}"
                       for n, df in TABLES.items())
    tools = [{
        "name": "lookup_rows",
        "description": (
            "Look up rows in one of the company's data tables by matching a column to a "
            "value. Use this to fetch real records before answering any question about a "
            "specific person, order, item, booking, account, etc."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "table_name": {"type": "string", "description": "Table to search. One of: " + ", ".join(TABLES)},
                "column":     {"type": "string", "description": "The column to match on."},
                "value":      {"type": "string", "description": "The value to look for in that column."},
            },
            "required": ["table_name", "column", "value"],
        },
    }]
else:
    schema = "  (no data tables provided)"
    tools = []

print("Data tables the copilot can query:")
print(schema)

## 7. Assemble the copilot — the whole thing in one place

Here's the full copilot in a single function. For each question it:

1. **Retrieves** the most relevant document chunks (RAG),
2. Hands Claude those excerpts **and** the `lookup_rows` tool, then
3. Lets Claude decide: answer from the documents, or look up live data — running any tool
   calls and looping until it has a final answer.

One combined loop, so there's no separate "router" to maintain — Claude routes itself.

In [ ]:
import anthropic, numpy as np, json
client = anthropic.Anthropic()

SYSTEM = f"""You are a helpful assistant for {COMPANY_NAME}. {COMPANY_DESCRIPTION}

Answer questions using ONLY the information available to you:
1. Company documents - provided as excerpts in the conversation (policies, FAQs, guides).
2. Company data tables - which you query with the lookup_rows tool. Available tables:
{schema}

Guidelines:
- For policy / how-to / FAQ questions, rely on the document excerpts and quote the relevant rule.
- For questions about a specific record, call lookup_rows to get real data before answering -
  never invent details like a status, price, quantity, or date.
- If the answer isn't in the documents or the data, say so honestly.
- Be concise and friendly."""

def retrieve(question, top_k=TOP_K):
    if index is None or not CHUNKS:
        return []
    q = embedder.encode([question], normalize_embeddings=True)
    _, ids = index.search(np.array(q, dtype="float32"), min(top_k, len(CHUNKS)))
    return [CHUNKS[i] for i in ids[0]]

def copilot(question, history=None, verbose=True):
    chunks = retrieve(question)
    context = "\n\n".join(f"[{c['source']}]\n{c['text']}" for c in chunks)
    opening = question if not context else (
        f"Relevant company documents:\n\n{context}\n\n---\n\nQuestion: {question}"
    )
    # Prepend the recent conversation so follow-up questions have context.
    messages = list(history or [])[-10:] + [{"role": "user", "content": opening}]

    while True:
        kwargs = dict(model=MODEL, max_tokens=MAX_TOKENS, system=SYSTEM, messages=messages)
        if tools:
            kwargs["tools"] = tools
        response = client.messages.create(**kwargs)

        if response.stop_reason != "tool_use":          # Claude's final answer
            return "".join(b.text for b in response.content if b.type == "text")

        messages.append({"role": "assistant", "content": response.content})
        results = []
        for block in response.content:
            if block.type == "tool_use":
                output = TOOL_FUNCTIONS[block.name](**block.input)
                if verbose:
                    print(f"  🔧 {block.name}({block.input})")
                results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps(output, default=str),
                })
        messages.append({"role": "user", "content": results})

print("Copilot ready. Ask it something below. 🏔️")

## 8. Ask your copilot  ✏️ TRY IT

Change the question and run this cell. Try a **policy question** (answered from your
documents) and a **record question** (answered by looking up your data) to see both paths.
The 🔧 lines show when Claude reaches into your tables.

In [ ]:
QUESTION = "What is your return policy?"      # ✏️ ask your copilot anything

print(copilot(QUESTION))

## 9. Deploy your copilot as a web app 🚀

Let's give *your* copilot a real interface. With **Gradio** we wrap the `copilot` function
in a chat web app in a few lines. `.launch(share=True)` prints a public link
(`https://xxxx.gradio.live`) you can open on your phone or share with someone — it stays
live while this notebook is running.

💬 The app also **remembers the conversation**, so people can ask natural follow-up questions.

> 🔒 **Heads up:** the share link is **public and temporary** (it expires in ~72 hours), and
> anyone with it can chat using *your* API credit. Keep it to yourself, and stop this cell to
> take the app down.

In [ ]:
%pip install -q gradio

In [ ]:
import gradio as gr

def to_messages(history):
    """Normalize Gradio chat history to Claude-style [{"role","content"}, ...].
    Works with both the newer 'messages' format and older [user, bot] pairs, so this
    runs regardless of which Gradio version Colab has installed."""
    if not history:
        return []
    msgs = []
    for item in history:
        if isinstance(item, dict):
            msgs.append({"role": item["role"], "content": item["content"]})
        else:                       # older Gradio: [user_text, bot_text] pair
            user, assistant = item
            if user is not None:
                msgs.append({"role": "user", "content": user})
            if assistant is not None:
                msgs.append({"role": "assistant", "content": assistant})
    return msgs

def chat(message, history):
    # `history` is the running conversation; copilot uses it so follow-ups work.
    return copilot(message, history=to_messages(history), verbose=False)

demo = gr.ChatInterface(
    fn=chat,
    title=f"{COMPANY_NAME} — Copilot",
    description="Ask about our policies, or look up a specific record.",
    examples=["What is your return policy?"],
)

demo.launch(share=True)   # in Colab this prints a public https://xxxx.gradio.live link

## 10. Where to go next

- **Improve answers:** edit `COMPANY_DESCRIPTION` and the guidelines in `SYSTEM` (Section 7)
  to shape tone and rules — then re-ask. Change one thing at a time and compare (Notebook 6).
- **Add knowledge:** drop more documents in `my_docs/` or more tables in `my_data/` and
  re-run Sections 4–6. No other code changes needed.
- **Evaluate it:** reuse the evaluation + LLM-as-judge approach from Notebook 6 to measure
  quality as you tweak.
- **Watch cost:** stick with `claude-haiku-4-5` and a modest `MAX_TOKENS`; the whole thing
  runs for a few cents.

That's a working, custom AI copilot for *your* business — nice work! 🎉